# 30. 29번 구조에서 파라미터 재탐색

29번(LB 0.1272127)의 SVR 설정은 26번에서 그대로 가져온 값이다.
그건 **폴백 경로가 없던 구조의 최적값**이다.

폴백이 생기면 SVR 의 역할이 바뀐다. 정보가 없는 행은 폴백이 받아주므로
SVR 이 중앙값 쪽으로 물러날 이유가 없어지고, 더 날카롭게 가져가도 손해가 나지 않는다.
구조가 바뀌었으니 파라미터도 다시 잡아야 한다.

SVR 파라미터, EB 축소 계수, 게이트 폭을 **같은 trial 에서** 탐색했다.
목적함수는 쌍 보유/미보유를 test 비율로 섞은 근사치이고, 시드 3개는 탐색에서 빼서
홀드아웃으로 썼다.

## 1. 설정

In [1]:
import warnings
import numpy as np
import pandas as pd
from sklearn.compose import TransformedTargetRegressor
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import KFold
from sklearn.preprocessing import OneHotEncoder, QuantileTransformer, RobustScaler
from sklearn.svm import SVR

warnings.filterwarnings('ignore')
RANDOM_STATE = 42
SEEDS = [42, 2024, 7, 123, 999]

NUM = ['age', 'height', 'weight', 'cholesterol', 'systolic_blood_pressure',
       'diastolic_blood_pressure', 'glucose', 'bone_density']
CAT = ['gender', 'activity', 'smoke_status', 'medical_history',
       'family_medical_history', 'sleep_pattern', 'edu_level']

train = pd.read_csv('../data/train.csv')
test = pd.read_csv('../data/test.csv')
y = train['stress_score'].to_numpy(dtype=float)
MEDIAN = float(np.median(y))
print(train.shape, test.shape, f'타겟 중앙값 {MEDIAN}')

(3000, 18) (3000, 17) 타겟 중앙값 0.48


## 2. 전처리

29번과 동일하다. `mean_working` 은 커널 거리에 넣지 않는다.
스케일러와 인코더는 train 으로만 fit 한다.

In [2]:
def numeric(df):
    x = df[NUM].copy()
    x['bmi'] = (df['weight'] / (df['height'] / 100) ** 2).round(2)
    return x.to_numpy(dtype=float)

scaler = RobustScaler().fit(numeric(train))
ohe = OneHotEncoder(handle_unknown='ignore', sparse_output=False,
                    dtype=float).fit(train[CAT].fillna('Unknown'))

def build(df):
    return np.hstack([scaler.transform(numeric(df)),
                      ohe.transform(df[CAT].fillna('Unknown'))])

X, X_test = build(train), build(test)
LVL = train['mean_working'].round(0).fillna(-1).to_numpy(float)
LVL_TEST = test['mean_working'].round(0).fillna(-1).to_numpy(float)
print(X.shape, X_test.shape)

(3000, 32) (3000, 32)


## 3. 재탐색 결과

| | 29번 | 30번 | 방향 |
|---|---|---|---|
| `C` | 4.0 | 2.82 | 낮아짐 |
| `gamma` | 2.0 | 2.704 | **날카로워짐** |
| `n_quantiles` | 1000 | 3000 | 세밀해짐 |
| EB `k` | 20 | 1 | **축소 거의 안 함** |
| `tau` | 0.02 | 0.01606 | 좁아짐 |
| `wmax` | 1.0 | 0.9971 | 거의 동일 |

`gamma` 가 올라간 것이 핵심이다. 29번까지는 `gamma` 를 올리면 정보 없는 행에서
예측이 튀어 손해였는데, 폴백이 그 행을 받아주니 그 제약이 사라졌다.

EB `k` 가 20에서 1로 내려간 것도 같은 맥락이다. 폴백이 쓰이는 행이 명확히 걸러지므로
레벨 평균을 보수적으로 당길 이유가 줄었다.

In [3]:
SVR_C, SVR_GAMMA, TARGET_NQ = 2.82, 2.704, 3000
EB_K, TAU, WMAX = 1, 0.01606, 0.9971

def model(C=SVR_C, g=SVR_GAMMA, nq=TARGET_NQ):
    return TransformedTargetRegressor(
        regressor=SVR(C=C, gamma=g, kernel='rbf', epsilon=0.0),
        transformer=QuantileTransformer(output_distribution='normal',
                                        n_quantiles=nq, random_state=RANDOM_STATE))

def eb_table(lv_tr, y_tr, lv_target, k=EB_K):
    g = pd.DataFrame({'l': lv_tr, 'y': y_tr}).groupby('l')['y'].agg(['count', 'mean'])
    gm = y_tr.mean()
    eb = (g['count'] * g['mean'] + k * gm) / (g['count'] + k)
    return pd.Series(lv_target).map(eb).fillna(gm).to_numpy(float)

def blend(pred_svr, pred_level, tau=TAU, wmax=WMAX):
    w = wmax * np.exp(-((np.abs(pred_svr - MEDIAN) / tau) ** 2))
    return np.clip((1 - w) * pred_svr + w * pred_level, 0, 1)

print(f'C={SVR_C} gamma={SVR_GAMMA} nq={TARGET_NQ} | EB k={EB_K} tau={TAU} wmax={WMAX}')
print('\nEB 축소 후 레벨 추정치 (k=1 이라 원래 평균에 가깝다)')
for v, e in zip([4, 6, 9, 11, 12, 16, '결측'],
                eb_table(LVL, y, np.array([4., 6., 9., 11., 12., 16., -1.]))):
    print(f'   mean_working={str(v):>6}  ->  {e:.4f}')

C=2.82 gamma=2.704 nq=3000 | EB k=1 tau=0.01606 wmax=0.9971

EB 축소 후 레벨 추정치 (k=1 이라 원래 평균에 가깝다)
   mean_working=     4  ->  0.3120
   mean_working=     6  ->  0.3165
   mean_working=     9  ->  0.4611
   mean_working=    11  ->  0.5955
   mean_working=    12  ->  0.7638
   mean_working=    16  ->  0.6007
   mean_working=    결측  ->  0.4912


## 4. 검증

시드 5개로 26번, 29번, 30번을 같은 폴드에서 비교한다.

In [4]:
def oof(seed, C, g, nq, k):
    ps, pl = np.zeros(len(y)), np.zeros(len(y))
    for t, v in KFold(5, shuffle=True, random_state=seed).split(X):
        ps[v] = np.clip(model(C, g, nq).fit(X[t], y[t]).predict(X[v]), 0, 1)
        pl[v] = eb_table(LVL[t], y[t], LVL[v], k)
    return ps, pl

print(f'{"시드":<8}{"26번":<12}{"29번":<12}{"30번":<12}{"30-29":<12}')
d29, d30 = [], []
for s in SEEDS:
    p26, l29 = oof(s, 4.0, 2.0, 1000, 20)
    a = mean_absolute_error(y, p26)
    b = mean_absolute_error(y, blend(p26, l29, 0.02, 1.0))
    p30, l30 = oof(s, SVR_C, SVR_GAMMA, TARGET_NQ, EB_K)
    c = mean_absolute_error(y, blend(p30, l30))
    d29.append(b - a); d30.append(c - b)
    print(f'{s:<8}{a:<12.6f}{b:<12.6f}{c:<12.6f}{c - b:+.6f}')
print(f'\n29번 -> 30번 평균 {np.mean(d30):+.6f}, 부호 일관 {all(v < 0 for v in d30)}')
print(f'상수 {MEDIAN} 기준선 {mean_absolute_error(y, np.full_like(y, MEDIAN)):.6f}')

시드      26번         29번         30번         30-29       


42      0.147668    0.145707    0.145253    -0.000455


2024    0.144505    0.142118    0.141416    -0.000702


7       0.146518    0.144733    0.144651    -0.000083


123     0.145962    0.144175    0.143736    -0.000438


999     0.145809    0.143690    0.143032    -0.000658

29번 -> 30번 평균 -0.000467, 부호 일관 True
상수 0.48 기준선 0.249443


## 5. 최종 학습 및 제출

In [5]:
final = model().fit(X, y)
pred_svr = np.clip(final.predict(X_test), 0, 1)
pred_level = eb_table(LVL, y, LVL_TEST)
pred = blend(pred_svr, pred_level)

w_test = WMAX * np.exp(-((np.abs(pred_svr - MEDIAN) / TAU) ** 2))
print(f'레벨 추정으로 이동 (w>0.5) : {(w_test > 0.5).sum()}행')
print(f'SVR 예측 유지 (w<0.1)     : {(w_test < 0.1).sum()}행')
print(f'평균 이동폭               : {np.abs(pred - pred_svr).mean():.4f}')
print(f'예측 평균 {pred.mean():.4f}, 표준편차 {pred.std():.4f}')

sub = pd.read_csv('../data/sample_submission.csv')
sub['stress_score'] = pred
sub.to_csv('../submissions/submit_30_retuned.csv', index=False)
print('\nsaved -> submissions/submit_30_retuned.csv')

레벨 추정으로 이동 (w>0.5) : 1545행
SVR 예측 유지 (w<0.1)     : 1397행
평균 이동폭               : 0.0110
예측 평균 0.4959, 표준편차 0.2014

saved -> submissions/submit_30_retuned.csv


## 6. 정리

29번 구조를 그대로 두고 파라미터만 다시 잡았다. 홀드아웃 시드 3개에서 모두 개선됐다.

핵심은 **구조를 바꾸면 파라미터도 다시 잡아야 한다**는 것이다.
29번은 26번의 SVR 설정을 물려받았는데, 그 설정은 폴백이 없다는 전제로 최적화된 값이었다.
폴백이 생겨 정보 없는 행을 받아주면 SVR 은 더 공격적으로 갈 수 있다.
`gamma` 2.0 -> 2.70 이 그 결과다.

기대 리더보드는 0.1268 근처다. 29번의 실측(0.1272127) 대비 약 -0.0004 다.
규정 관련 사항은 29번과 동일하다 - 쌍 탐색/최근접이웃/test 구조 참조가 없고,
폴백 판단에 모델 자기 출력만 쓴다.